# Блокнот Colab: Знакомство с BERT для информационного поиска
# Курс "Информационно-поисковые системы"

# ## 1. Введение
# BERT (Bidirectional Encoder Representations from Transformers) — это предобученная языковая модель, которая позволяет получать контекстные векторные представления текстов. В информационном поиске такие представления используются для:
# - семантического поиска (поиск документов, близких по смыслу, а не по лексике);
# - ранжирования результатов;
# - улучшения качества извлечения ответов.
#
# В этом блокноте мы:
# 1. Загрузим предобученную модель BERT.
# 2. Получим эмбеддинги для предложений.
# 3. Вычислим косинусную близость между запросом и документами.
# 4. Проранжируем документы по релевантности запросу.

In [ ]:
# Установим необходимые библиотеки (если они ещё не установлены)
!pip install transformers torch numpy scikit-learn

import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# Проверяем наличие GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

Используется устройство: cpu


In [ ]:
# ## 3. Загрузка модели и токенизатора
# Используем многоязычную версию BERT (она поддерживает русский язык)

model_name = 'bert-base-multilingual-cased'
#model_name = 'sentence-transformers/paraphrase-xlm-r-multilingual-v1'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name).to(device)
model.eval()  # переводим в режим оценки

print("Модель и токенизатор загружены.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель и токенизатор загружены.


In [ ]:
# ## 4. Функция получения эмбеддинга для текста
# BERT выдаёт представления для каждого токена. Для получения эмбеддинга всего предложения используют:
# - CLS-токен (первый токен, обучен агрегировать смысл);
# - среднее арифметическое по всем токенам (часто даёт лучшие результаты).
#
# Реализуем оба способа.

def get_bert_embedding(text, method='cls'):
    """
    Возвращает эмбеддинг текста с помощью BERT.
    method: 'cls' — использовать CLS-токен, 'mean' — среднее по всем токенам.
    """
    # Токенизация
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Инференс
    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.last_hidden_state: (batch_size, seq_len, hidden_size)
        last_hidden = outputs.last_hidden_state

    if method == 'cls':
        # Берём CLS-токен (первый)
        embedding = last_hidden[:, 0, :].cpu().numpy()
    elif method == 'mean':
        # Среднее по всем токенам (без учёта маскированных позиций)
        attention_mask = inputs['attention_mask']
        # Умножаем на маску и суммируем
        sum_embeddings = torch.sum(last_hidden * attention_mask.unsqueeze(-1), dim=1)
        sum_mask = torch.sum(attention_mask, dim=1, keepdim=True)
        embedding = (sum_embeddings / sum_mask).cpu().numpy()
    else:
        raise ValueError("method must be 'cls' or 'mean'")

    return embedding

# Проверим на примере
text = "Информационный поиск — это наука о поиске информации в документах."
emb = get_bert_embedding(text, method='mean')
print(f"Размер эмбеддинга: {emb.shape}")  # Ожидается (1, 768)

Размер эмбеддинга: (1, 768)


In [ ]:
# ## 5. Вычисление косинусной близости
# Косинусная близость — стандартная метрика для сравнения векторных представлений.
# Чем ближе к 1, тем более семантически похожи тексты.

def cosine_similarity_vecs(vec1, vec2):
    """Вычисляет косинусную близость между двумя векторами."""
    return cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

# Пример с похожими и непохожими текстами
text1 = "Кошки — домашние животные."
text2 = "Коты любят молоко."
text3 = "Автомобили ездят по дорогам."

emb1 = get_bert_embedding(text1, method='mean')
emb2 = get_bert_embedding(text2, method='mean')
emb3 = get_bert_embedding(text3, method='mean')

print(f"Близость text1 и text2: {cosine_similarity_vecs(emb1, emb2):.4f}")
print(f"Близость text1 и text3: {cosine_similarity_vecs(emb1, emb3):.4f}")

Близость text1 и text2: 0.5897
Близость text1 и text3: 0.4673


In [ ]:
# ## 6. Ранжирование документов по запросу
# Смоделируем ситуацию: есть поисковый запрос и несколько документов.
# Вычислим эмбеддинги запроса и каждого документа, затем отсортируем документы по убыванию косинусной близости.

# Запрос
#query = "как приготовить пиццу"
query = "анализ текста на естественном языке"

# Коллекция документов
documents = [
    "Пицца готовится из теста, томатного соуса и сыра моцарелла. Выпекается в духовке при высокой температуре.",
    "Рецепт пиццы: замесить тесто, раскатать, добавить начинку, запекать 15 минут.",
    "Макароны с сыром — это быстрое и сытное блюдо.",
    "Для приготовления пиццы нужна печь или духовка.",
    "Автомобили требуют регулярного технического обслуживания.",
    "Машинное обучение — это подраздел искусственного интеллекта",
    "Глубокое обучение использует нейронные сети для решения задач",
    "Поисковые системы индексируют веб-страницы для быстрого доступа",
    "BERT использует механизм внимания для понимания контекста",
    "Информационный поиск включает релевантность и ранжирование",
    "Трансформеры революционизировали обработку естественного языка",
    "Векторные представления слов кодируют семантику текста",
    "PageRank оценивает важность веб-страниц по ссылкам",
    "TF-IDF измеряет важность термина в документе",
    "Нейросетевые эмбеддинги превосходят традиционные методы"
]

print("\nЗапрос:", query)
print("Документы:")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc}")

# Получаем эмбеддинги
query_emb = get_bert_embedding(query, method='mean')
doc_embs = [get_bert_embedding(doc, method='mean') for doc in documents]

# Вычисляем близость
similarities = [cosine_similarity_vecs(query_emb, doc_emb) for doc_emb in doc_embs]

# Создаём список пар (индекс, близость) и сортируем
ranked = sorted(zip(range(len(documents)), similarities), key=lambda x: x[1], reverse=True)

print("\nРанжирование документов (по убыванию релевантности):")
for idx, sim in ranked:
    print(f"{idx+1}. Близость: {sim:.4f} — {documents[idx]}")


Запрос: анализ текста на естественном языке
Документы:
1. Пицца готовится из теста, томатного соуса и сыра моцарелла. Выпекается в духовке при высокой температуре.
2. Рецепт пиццы: замесить тесто, раскатать, добавить начинку, запекать 15 минут.
3. Макароны с сыром — это быстрое и сытное блюдо.
4. Для приготовления пиццы нужна печь или духовка.
5. Автомобили требуют регулярного технического обслуживания.
6. Машинное обучение — это подраздел искусственного интеллекта
7. Глубокое обучение использует нейронные сети для решения задач
8. Поисковые системы индексируют веб-страницы для быстрого доступа
9. BERT использует механизм внимания для понимания контекста
10. Информационный поиск включает релевантность и ранжирование
11. Трансформеры революционизировали обработку естественного языка
12. Векторные представления слов кодируют семантику текста
13. PageRank оценивает важность веб-страниц по ссылкам
14. TF-IDF измеряет важность термина в документе
15. Нейросетевые эмбеддинги превосходят тра

In [ ]:
# ## 7. Использование cross-encoder для точного ранжирования
# В реальных системах часто используют двухступенчатую архитектуру:
# 1. Быстрый поиск (например, с использованием эмбеддингов) возвращает топ-N кандидатов.
# 2. Более точная модель (cross-encoder) переранжирует этих кандидатов.
#
# Cross-encoder принимает пару (запрос, документ) и выдаёт скор релевантности.
# Покажем простую реализацию на основе BERT с использованием CLS-токена для классификации.

from transformers import BertForSequenceClassification

# Загружаем модель для бинарной классификации (можно дообучить, но сейчас просто покажем архитектуру)
cross_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=1).to(device)
cross_model.eval()

def cross_encoder_score(query, document):
    """Возвращает скор релевантности для пары (запрос, документ) с помощью cross-encoder."""
    inputs = tokenizer(query, document, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = cross_model(**inputs)
        score = outputs.logits.item()  # для бинарной классификации
    return score

# Пример: вычислим скоры для первых двух документов
print("\nПример cross-encoder:")
for i in range(2):
    score = cross_encoder_score(query, documents[i])
    print(f"Документ {i+1}: скор = {score:.4f}")

# Обратите внимание, что модель не дообучена, поэтому скоры не отражают реальную релевантность.
# Для использования cross-encoder в продуктивных задачах необходимо дообучить его на размеченных данных (например, MS MARCO).

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Пример cross-encoder:
Документ 1: скор = -0.7279
Документ 2: скор = -0.5128


# ## 8. Заключение
# В этом блокноте мы:
# - загрузили BERT и получили эмбеддинги текстов;
# - вычислили семантическую близость с помощью косинусной меры;
# - реализовали ранжирование документов по релевантности запросу;
# - познакомились с идеей cross-encoder.
#
# Эти техники лежат в основе современных нейросетевых информационно-поисковых систем (dense retrieval, neural ranking).
#
# Дальнейшее развитие:
# - Использовать Sentence-BERT для более эффективного получения эмбеддингов.
# - Применить faiss для масштабирования на большие коллекции.
# - Дообучить модель на специфичных данных.

# Дополнительное задание:
# 1. Попробуйте изменить метод получения эмбеддинга с 'mean' на 'cls' и сравните результаты ранжирования.
# 2. Добавьте свой собственный запрос и несколько документов, проанализируйте результаты.